# Lab 21 — Fine-tuning LLMs · RUN ALL & DEMO (T4 GPU)

Chạy từ trên xuống dưới trên Google Colab (**Runtime > Change runtime type > T4 GPU**).

| Ô | Mục đích | Thời gian ước tính |
|---|---|---|
| 1 | **Setup** (Clone repo + Cài đặt dependencies) | ~1 phút |
| 2 | **Smoke Test** (Kiểm tra imports, dataset, unit tests) | ~30 giây |
| 3 | **Core Pipeline + NB6** (NB1 -> NB5 + NB6) | ~90–120 phút |
| 4 | **Gatekeeper Verify** (Kiểm tra liêm chính và artefact) | ~10 giây |
| 5 | **Bonus Challenges** (B1 Merge, B4 Rank Sweep, B5 HF Hub) | ~15 phút |
| 6 | **Interactive Web Demo** (Khởi chạy Gradio Demo trực quan) | Tức thì |
| 7 | **Tải File Nộp Bài (ZIP)** (Đóng gói results, adapter, report) | ~10 giây |


In [ ]:
# @title 1. Setup — Clone Repo & Cài đặt Thư viện (Chạy ô này đầu tiên)
import os, subprocess, sys

REPO = "https://github.com/Lolavine777/Day21-Track3-Finetuning-Lab-2A202601934-NguyenDangLong.git"
REPO_DIR = os.path.basename(REPO).replace(".git", "")

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir(f"/content/{REPO_DIR}" if os.path.exists(f"/content/{REPO_DIR}") else REPO_DIR)
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install requirements
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)
# Install gradio for interactive demo
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"],
               check=True)

import torch
print("Commit :", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Hãy vào Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke Test — Kiểm tra môi trường & 118 Unit Tests
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Chạy Core Pipeline (NB1 → NB5 + NB6)
# Lưu ý: EVAL_LIMIT để rỗng "" để chạy full eval set nộp bài (submittable run)
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""          # @param ["", "4", "8", "16", "25"]
STAGES       = "all"       # @param {type:"string"}
EPOCHS       = "2"         # @param ["1", "2", "3"]

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
os.environ["EPOCHS"] = EPOCHS
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper Verify — Thẩm định toàn bộ Artefacts trước khi nộp
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null
!echo && echo "---- autopsy.json ----" && cat results/autopsy.json 2>/dev/null
!echo && echo "---- merge_check.json ----" && cat results/merge_check.json 2>/dev/null


In [ ]:
# @title 5. Bonus Challenges (B4 Rank Sweep & B5 HF Hub)
# 5.1 Chạy Bonus B4: Quét Rank có kiểm soát r in {8, 16, 64}
!python scripts/run_bonus_b4.py

# 5.2 (Tuỳ chọn) Push adapter lên HuggingFace Hub (Bonus B5 +2đ)
# Bỏ comment và điền token + repo_id của bạn để đẩy adapter lên Hub:
# !python scripts/push_to_hub.py --repo-id "your-username/lab21-qwen35-triage-vi" --token "hf_xxx"


In [ ]:
# @title 6. Khởi chạy Interactive Web Demo (Gradio Live Share)
# Ô này sẽ mở một link public gradio.live để bạn demo trực tiếp trên trình duyệt!
!python demo/gradio_app.py


In [ ]:
# @title 7. Tự động đóng gói ZIP nộp bài & Tải về máy (Option A / Rubric)
import shutil
from google.colab import files

ZIP_NAME = "lab21_2A202601934.zip"
!zip -q -r {ZIP_NAME} results/ adapters/correct/ notebooks/ submission/
print(f"Đã tạo gói nộp bài: {ZIP_NAME}")
files.download(ZIP_NAME)
